https://www.sgx.com/securities/etf-screener

# Setup

In [1]:
# Setup
%load_ext autoreload
%autoreload 2
#%load_ext jupyter_ai_magics
import sys; sys.path.append('..')
from src.env_setup import init_analysis_env; init_analysis_env()


df = load_local_data("myData*.csv*", na_values=['-'], dtype={'Code': str})
df.head()

# Cleaning

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
🚀 Analysis environment initialized: pd, np, yf, plt, sns, load_local_data, datetime, timedelta, xw are ready.
Loading: /home/mrsmmori/notebooks/Win_Downloads/myData.csv


,Trading Name,Trading Code,CCY,Underlying Benchmark,Close,Last,Chng %,Val ($M),Yield (%),TER (%),...,Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Sustainability linked,Management Style,CPF Eligibility,SIP/EIP,Tick Size
0,STI ETF,ES3,SGD,Straits Times Index,4.41,4.41,0.14,5.87,4.17,0.28,...,30.48,14.80,Equities,Singapore,Distributing,No,PASSIVE,Yes,EIP,0.00
1,IS ASIA HYG US$,O9P,USD,Bloomberg Asia USD High Yield Diversified Cred...,6.70,6.70,0.30,4.54,7.23,0.50,...,7.16,8.99,Fixed Income,Asia,Distributing,No,PASSIVE,No,EIP,0.01
2,GLD US$,O87,USD,LBMA Gold Price PM,340.00,340.00,1.52,4.10,NaN,0.40,...,35.65,24.62,Commodities,Gold,NaN,No,PASSIVE,Yes,EIP,0.01
3,Lion-OCBC Sec HSTECH S$,HST,SGD,Hang Seng TECH Index,0.96,0.96,1.26,4.09,NaN,0.58,...,61.74,7.35,Equities,China,Accumulating,No,PASSIVE,No,EIP,0.00
4,LION-PHILLIP S-REIT,CLR,SGD,Morningstar Singapore REIT Yield Focus Index SM,0.86,0.86,0.93,2.97,5.67,0.60,...,-0.23,-1.81,REITs,Singapore,Distributing,No,PASSIVE,No,EIP,0.00


In [2]:
target_columns = [
    "Yield (%)",
    "TR 1M (%)",
    "TR 3M (%)",
    "TR 1Y (%)",
    "Ann. TR 1Y (%)",
    "Ann. TR 3Y (%)",
]

df[target_columns] = df[target_columns].round(2)

In [3]:
df["Geographical Focus"].value_counts()

Geographical Focus
China               29
Singapore           17
Asia                15
Asia Pacific         8
India                3
Vietnam              3
Gold                 2
USA                  2
Japan                2
Saudi Arabia         1
Indonesia            1
Emerging Markets     1
Name: count, dtype: int64

# df -> df_usd, df_sgd

In [4]:
df_usd = df[(df["CCY"] == "USD") | (df["CCY"] == "USD")].copy()
df_sgd = df[(df["CCY"] == "SGD") | (df["CCY"] == "SGD")].copy()

# df_sgd -> df

In [5]:
df = df_sgd.copy()
df = df[df["Sustainability linked"] == "No"]

In [6]:
df["Geographical Focus"].value_counts()

Geographical Focus
China           13
Singapore        9
Asia             6
Asia Pacific     2
Gold             1
Japan            1
Vietnam          1
Name: count, dtype: int64

In [7]:
df["Asset Class"].value_counts()

Asset Class
Equities        20
Fixed Income     8
REITs            4
Commodities      1
Name: count, dtype: int64

In [8]:
df["Income Treatment"].value_counts()

Income Treatment
Distributing    20
Accumulating    12
Name: count, dtype: int64

In [9]:
base_url = "https://investors.sgx.com/market/securities?code="
last_url = "&type=etfs&lang=en"
df['URL'] = base_url + df['Trading Code'] + last_url

In [10]:
df = df[
    [
        "Trading Name",
        "Trading Code",
        #'CCY',
        "Underlying Benchmark",
        "URL",
        #'Close',
        #'Last',
        #'Chng %',
        #'Val ($M)',
        "Yield (%)",
        #'TER (%)',
        #'Documents',
        #'Fund Manager',
        "TR 1M (%)",
        "TR 3M (%)",
        "TR 1Y (%)",
        "Ann. TR 1Y (%)",
        "Ann. TR 3Y (%)",
        "Asset Class",
        "Geographical Focus",
        "Income Treatment",
        #'Sustainability linked',
        "Management Style",
        #'CPF Eligibility',
        #'SIP/EIP',
        #'Tick Size',
    ]
]

In [11]:
pd.crosstab(df["Asset Class"], df["Income Treatment"])

Income Treatment,Accumulating,Distributing
Asset Class,,
Equities,11,9
Fixed Income,1,7
REITs,0,4


In [12]:
df_bonds = df.query("`Asset Class` == 'BONDS'")
df_bonds.style.format({'URL': lambda x: f'<a target="_blank" href="{x}">link</a>'})

,Trading Name,Trading Code,Underlying Benchmark,URL,Yield (%),TR 1M (%),TR 3M (%),TR 1Y (%),Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Management Style


In [13]:
df_equities = df.query("`Asset Class`.str.lower() == 'equities'")
df_equities.style.format({'URL': lambda x: f'<a target="_blank" href="{x}">link</a>'})

,Trading Name,Trading Code,Underlying Benchmark,URL,Yield (%),TR 1M (%),TR 3M (%),TR 1Y (%),Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Management Style
0,STI ETF,ES3,Straits Times Index,link,4.170000,2.490000,10.480000,30.480000,30.480000,14.800000,Equities,Singapore,Distributing,PASSIVE
3,Lion-OCBC Sec HSTECH S$,HST,Hang Seng TECH Index,link,nan,0.550000,6.940000,61.740000,61.740000,7.350000,Equities,China,Accumulating,PASSIVE
9,Amova STI ETF,G3B,Straits Times Index,link,4.090000,2.500000,10.670000,31.110000,31.110000,14.810000,Equities,Singapore,Distributing,PASSIVE
10,Lion-OSPL China L S$,YYY,Hang Seng Stock Connect China 80 Index,link,3.050000,1.520000,7.980000,32.610000,32.610000,5.940000,Equities,China,Distributing,PASSIVE
13,CSOP Star&Chinext50 S$,SCY,CSI STAR and ChiNext 50 Index,link,nan,nan,nan,nan,nan,nan,Equities,China,Accumulating,PASSIVE
14,XT MSCHINA S$,TID,MSCI China TR Net Daily USD Index,link,nan,1.080000,10.760000,45.530000,45.530000,5.820000,Equities,China,Accumulating,PASSIVE
15,Amova Asia exJC S$,A93,MSCI AC Asia ex Japan ex China Index,link,nan,nan,nan,nan,nan,nan,Equities,Asia,Accumulating,PASSIVE
17,Lion-OSPL APAC Fin S$,YLD,iEdge APAC Financials Dividend Plus Index,link,6.440000,4.430000,5.850000,17.940000,17.940000,nan,Equities,Asia Pacific,Distributing,PASSIVE
22,UOBAM FTSE CN A50 S$,JK8,FTSE China A50 Index,link,nan,nan,nan,nan,nan,nan,Equities,China,Distributing,PASSIVE
30,AmovaEFund ChiNext S$,CXT,ChiNext Index,link,nan,nan,nan,nan,nan,nan,Equities,China,Accumulating,PASSIVE


In [14]:
df_reits = df.query("`Asset Class` == 'REITs'")
df_reits.style.format({'URL': lambda x: f'<a target="_blank" href="{x}">link</a>'})

,Trading Name,Trading Code,Underlying Benchmark,URL,Yield (%),TR 1M (%),TR 3M (%),TR 1Y (%),Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Management Style
4,LION-PHILLIP S-REIT,CLR,Morningstar Singapore REIT Yield Focus Index SM,link,5.670000,0.360000,8.630000,-0.230000,-0.230000,-1.810000,REITs,Singapore,Distributing,PASSIVE
5,Amova-STC Asia REIT,CFA,FTSE EPRA/NAREIT Asia ex Japan Net Total Return REIT Index,link,5.450000,1.550000,9.090000,6.530000,6.530000,-1.400000,REITs,Asia,Distributing,PASSIVE
11,CSOP iEdge SREIT ETF S$,SRT,iEdge SREIT Leaders Index,link,5.730000,3.850000,11.220000,2.550000,2.550000,-1.130000,REITs,Singapore,Distributing,PASSIVE
23,PHIL AP DIV REIT S$D,BYJ,iEdge APAC ex Japan Dividend Leaders REIT Index,link,4.280000,nan,nan,nan,nan,nan,REITs,Asia Pacific,Distributing,PASSIVE


In [15]:
#df.style.format({'URL': lambda x: f'<a target="_blank" href="{x}">link</a>'})

In [16]:
code = "VND"
url = f"https://www.example.com/stocks/{code}"
print(url)

https://www.example.com/stocks/VND


# df_usd -> df

In [17]:
df = df_usd.copy()
df = df[df["Sustainability linked"] == "No"]

In [18]:
grouped = df.groupby(["Asset Class", "Income Treatment"])

for (asset_class, income), group_df in grouped:
    print(f"=== Asset Class: {asset_class}, Income Treatment: {income} ===")
    group_df

=== Asset Class: Equities, Income Treatment: Accumulating ===


,Trading Name,Trading Code,CCY,Underlying Benchmark,Close,Last,Chng %,Val ($M),Yield (%),TER (%),...,Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Sustainability linked,Management Style,CPF Eligibility,SIP/EIP,Tick Size
28,XT MSCHINA US$,LG9,USD,MSCI China TR Net Daily USD Index,21.00,21.00,1.79,0.03,NaN,0.65,...,47.90,8.73,Equities,China,Accumulating,No,PASSIVE,No,EIP,0.01
31,CF VN 30 SC ETF US$,VND,USD,iEdge Vietnam 30 Sector Cap USD Index (NTR),1.28,1.26,2.52,0.03,NaN,2.39,...,50.79,NaN,Equities,Vietnam,Accumulating,No,PASSIVE,No,EIP,0.00
38,Amundi MSIndia Sw US$,G1N,USD,MSCI India Net Total Return Index,32.10,32.23,0.41,0.00,NaN,0.85,...,-11.87,7.62,Equities,India,Accumulating,No,PASSIVE,No,SIP,0.01
41,XT MSINDO US$,KJ7,USD,MSCI Daily TRN Net Emerging Markets Indonesia ...,13.47,13.18,1.39,0.00,NaN,0.65,...,-16.77,-4.46,Equities,Indonesia,Accumulating,No,PASSIVE,No,SIP,0.01
46,XT Vietnam US$,HD9,USD,FTSE Vietnam Index,37.67,37.61,1.65,0.00,NaN,0.85,...,43.57,2.79,Equities,Vietnam,Accumulating,No,PASSIVE,No,SIP,0.01
49,A Lion-Nomura Japan US$,JUS,USD,Topix Index,0.97,0.97,0.00,0.00,NaN,0.98,...,30.13,NaN,Equities,Japan,Accumulating,No,ACTIVE,No,EIP,0.00
50,Amova Asia exJC US$,A94,USD,MSCI AC Asia ex Japan ex China Index,0.89,0.86,0.00,0.00,NaN,NaN,...,NaN,NaN,Equities,Asia,Accumulating,No,PASSIVE,No,EIP,0.00
57,AmovaEFund ChiNext US$,CXO,USD,ChiNext Index,0.19,0.00,0.00,0.00,NaN,NaN,...,NaN,NaN,Equities,China,Accumulating,No,PASSIVE,No,EIP,0.00
58,Amundi EM Mkt Sw US$,H1N,USD,MSCI Emerging Market Net Total Return Index,16.67,16.25,0.00,0.00,NaN,0.55,...,17.18,10.54,Equities,Emerging Markets,Accumulating,No,PASSIVE,No,SIP,0.01
63,CSOP SEA TECH ETF US$,SQU,USD,iEdge Southeast Asia+ TECH Index,0.96,0.93,0.00,0.00,NaN,1.41,...,13.89,NaN,Equities,Asia,Accumulating,No,PASSIVE,No,EIP,0.00


=== Asset Class: Equities, Income Treatment: Distributing ===


,Trading Name,Trading Code,CCY,Underlying Benchmark,Close,Last,Chng %,Val ($M),Yield (%),TER (%),...,Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Sustainability linked,Management Style,CPF Eligibility,SIP/EIP,Tick Size
20,SPDR S&P500 US$,S27,USD,S&P 500 Index,658.05,658.04,0.62,0.10,1.20,0.00,...,16.95,18.67,Equities,USA,Distributing,No,PASSIVE,No,SIP,0.01
40,Lion-OSPL APAC Fin US$,YLU,USD,iEdge APAC Financials Dividend Plus Index,0.99,0.99,3.22,0.00,6.44,1.08,...,44.26,NaN,Equities,Asia Pacific,Distributing,No,PASSIVE,No,EIP,0.00
60,CGS FG CSI1000 US$,GRU,USD,CSI 1000 Index,1.22,1.22,0.00,0.00,NaN,1.96,...,45.19,NaN,Equities,China,Distributing,No,PASSIVE,No,EIP,0.00
79,SPDR DJIA US$,D07,USD,Dow Jones Industrial AverageSM (DJIASM) Index,462.13,459.00,0.00,0.00,1.49,0.16,...,11.95,14.43,Equities,USA,Distributing,No,PASSIVE,No,SIP,0.01
81,UOBAM FTSE CN A50 US$,VK8,USD,FTSE China A50 Index,1.89,1.64,0.00,0.00,NaN,0.84,...,NaN,NaN,Equities,China,Distributing,No,PASSIVE,No,EIP,0.00


=== Asset Class: Fixed Income, Income Treatment: Accumulating ===


,Trading Name,Trading Code,CCY,Underlying Benchmark,Close,Last,Chng %,Val ($M),Yield (%),TER (%),...,Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Sustainability linked,Management Style,CPF Eligibility,SIP/EIP,Tick Size
36,SPDR JPM SaudiAggBd US$,KSB,USD,J.P. Morgan Saudi Arabia Aggregate Index,31.46,31.39,0.93,0.01,NaN,0.37,...,NaN,NaN,Fixed Income,Saudi Arabia,Accumulating,No,PASSIVE,No,EIP,0.00
53,Amova-ICBCSG CNB US$,ZHD,USD,ChinaBond ICBC 1-10 Year Treasury and Policy B...,0.84,0.83,0.00,0.00,NaN,0.30,...,NaN,NaN,Fixed Income,China,Accumulating,No,PASSIVE,No,EIP,0.00
64,ICBC CSOP CGB ETF US$A,CYX,USD,FTSE Chinese Government Bond Index,11.02,10.82,0.00,0.00,NaN,0.26,...,2.80,NaN,Fixed Income,China,Accumulating,No,PASSIVE,No,EIP,0.01


=== Asset Class: Fixed Income, Income Treatment: Distributing ===


,Trading Name,Trading Code,CCY,Underlying Benchmark,Close,Last,Chng %,Val ($M),Yield (%),TER (%),...,Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Sustainability linked,Management Style,CPF Eligibility,SIP/EIP,Tick Size
1,IS ASIA HYG US$,O9P,USD,Bloomberg Asia USD High Yield Diversified Cred...,6.70,6.70,0.30,4.54,7.23,0.50,...,7.16,8.99,Fixed Income,Asia,Distributing,No,PASSIVE,No,EIP,0.01
65,ICBC CSOP CGB ETF US$D,CYB,USD,FTSE Chinese Government Bond Index,10.06,10.05,0.00,0.00,2.95,0.26,...,3.29,3.07,Fixed Income,China,Distributing,No,PASSIVE,No,EIP,0.01
66,IS ASIA BND US$,N6M,USD,J.P. Morgan Asia Credit Index - Core,9.91,9.90,0.00,0.00,4.24,0.20,...,3.65,5.80,Fixed Income,Asia,Distributing,No,PASSIVE,No,EIP,0.01
78,PHILLIP MM US$D,MMT,USD,FTSE SGD 3-month SOR Index,80.61,79.76,0.00,0.00,2.88,0.28,...,NaN,NaN,Fixed Income,Singapore,Distributing,No,PASSIVE,No,EIP,0.00


=== Asset Class: REITs, Income Treatment: Distributing ===


,Trading Name,Trading Code,CCY,Underlying Benchmark,Close,Last,Chng %,Val ($M),Yield (%),TER (%),...,Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Sustainability linked,Management Style,CPF Eligibility,SIP/EIP,Tick Size
43,CSOP iEdge SREIT ETF US$,SRU,USD,iEdge SREIT Leaders Index,0.61,0.61,0.66,0.00,5.73,0.60,...,3.78,2.29,REITs,Singapore,Distributing,No,PASSIVE,No,EIP,0.00
54,Amova-STC A_REIT US$,COI,USD,FTSE EPRA/NAREIT Asia ex Japan Net Total Retur...,0.66,0.65,0.00,0.00,5.45,0.55,...,NaN,NaN,REITs,Asia,Distributing,No,PASSIVE,Yes,EIP,0.00
74,PHIL AP DIV REIT US$,BYI,USD,iEdge APAC ex Japan Dividend Leaders REIT Index,0.94,0.93,0.00,0.00,4.28,1.54,...,6.96,1.65,REITs,Asia Pacific,Distributing,No,PASSIVE,No,EIP,0.00


In [19]:
base_url = "https://investors.sgx.com/market/securities?code="
last_url = "&type=etfs&lang=en"
df['URL'] = base_url + df['Trading Code'] + last_url
df.style.format({'URL': lambda x: f'<a target="_blank" href="{x}">link</a>'})

,Trading Name,Trading Code,CCY,Underlying Benchmark,Close,Last,Chng %,Val ($M),Yield (%),TER (%),Documents,Fund Manager,TR 1M (%),TR 3M (%),TR 1Y (%),Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Sustainability linked,Management Style,CPF Eligibility,SIP/EIP,Tick Size,URL
1,IS ASIA HYG US$,O9P,USD,Bloomberg Asia USD High Yield Diversified Credit Index,6.700000,6.700000,0.299000,4.536544,7.230000,0.500000,Monthly Report (2025-08-12) goto https://www.sgx.com/assets/static/isin-documents/SG2D83975482/MR_SG_en_SG2D83975482_YES_2025-07-31.pdf Semi-annual Report (2025-07-28) goto https://www.sgx.com/assets/static/isin-documents/SG2D83975482/SAR_SG_en_SG2D83975482_YES_2024-06-30.pdf Annual Report (2024-08-14) goto https://www.sgx.com/assets/static/isin-documents/SG2D83975482/AR_SG_en_SG2D83975482_YES_2023-12-31.pdf,Black Rock Singapore (Ltd),1.180000,3.370000,7.160000,7.160000,8.990000,Fixed Income,Asia,Distributing,No,PASSIVE,No,EIP,0.010000,link
2,GLD US$,O87,USD,LBMA Gold Price PM,340.000000,340.000000,1.523000,4.095981,nan,0.400000,Monthly Report (2025-08-21) goto https://www.sgx.com/assets/static/isin-documents/US78463V1070/MR_SG_en_US78463V1070_YES_2025-07-31.pdf,State Street Investment Management,3.360000,3.420000,35.650000,35.650000,24.620000,Commodities,Gold,nan,No,PASSIVE,Yes,EIP,0.010000,link
20,SPDR S&P500 US$,S27,USD,S&P 500 Index,658.050000,658.040000,0.618000,0.103300,1.200000,0.000000,Monthly Report (2025-08-21) goto https://www.sgx.com/assets/static/isin-documents/US78462F1030/MR_SG_en_US78462F1030_YES_2025-07-31.pdf Semi-annual Report (2025-01-31) goto https://www.sgx.com/assets/static/isin-documents/US78462F1030/SAR_SG_en_US78462F1030_YES_2024-03-31.pdf Annual Report (2025-01-31) goto https://www.sgx.com/assets/static/isin-documents/US78462F1030/AR_SG_en_US78462F1030_YES_2024-09-30.pdf Prospectus (2023-02-23) goto https://www.sgx.com/assets/static/isin-documents/US78462F1030/PR_SG_en_US78462F1030_YES_2023-01-30.pdf,State Street Investment Management,1.520000,9.590000,16.950000,16.950000,18.670000,Equities,USA,Distributing,No,PASSIVE,No,SIP,0.010000,link
28,XT MSCHINA US$,LG9,USD,MSCI China TR Net Daily USD Index,21.000000,21.000000,1.794000,0.025956,nan,0.650000,Semi-annual Report (2025-08-29) goto https://www.sgx.com/assets/static/isin-documents/LU0514695690/SAR_SG_en_LU0514695690_YES_2025-06-30.pdf Annual Report (2025-03-28) goto https://www.sgx.com/assets/static/isin-documents/LU0514695690/AR_SG_en_LU0514695690_YES_2024-12-31.pdf Prospectus (2025-02-20) goto https://www.sgx.com/assets/static/isin-documents/LU0514695690/PR_SG_en_LU0514695690_YES_2025-02-19.pdf,DWS Investments,1.530000,11.160000,47.900000,47.900000,8.730000,Equities,China,Accumulating,No,PASSIVE,No,EIP,0.010000,link
31,CF VN 30 SC ETF US$,VND,USD,iEdge Vietnam 30 Sector Cap USD Index (NTR),1.276000,1.260000,2.522000,0.025200,nan,2.390000,nan,CGS International,13.680000,36.970000,50.790000,50.790000,nan,Equities,Vietnam,Accumulating,No,PASSIVE,No,EIP,0.001000,link
36,SPDR JPM SaudiAggBd US$,KSB,USD,J.P. Morgan Saudi Arabia Aggregate Index,31.465000,31.390000,0.932000,0.006278,nan,0.370000,nan,State Street Investment Management,nan,nan,nan,nan,nan,Fixed Income,Saudi Arabia,Accumulating,No,PASSIVE,No,EIP,0.001000,link
38,Amundi MSIndia Sw US$,G1N,USD,MSCI India Net Total Return Index,32.100000,32.230000,0.405000,0.004154,nan,0.850000,Monthly Report (2025-08-08) goto https://www.sgx.com/assets/static/isin-documents/FR0010375766/MR_SG_en_FR0010375766_YES_2025-07-31.pdf Annual Report (2025-06-15) goto https://www.sgx.com/assets/static/isin-documents/FR0010375766/AR_SG_en_FR0010375766_YES_2023-10-31.pdf Prospectus (2025-07-03) goto https://www.sgx.com/assets/static/isin-documents/FR0010375766/PR_SG_en_FR0010375766_YES_2025-06-20.pdf,Amundi Asset Management,-3.400000,-4.970000,-11.870000,-11.870000,7.620000,Equities,India,Accumulating,No,PASSIVE,No,SIP,0.010000,link
40,Lion-OSPL APAC Fin US$,YLU,USD,iEdge APAC Financials Di